# Label cell types using CellTypist Models

To build our reference, we would like to start with labels that originate from published cell type references. 

One of the approaches for this cell type labeling is CellTypist, a model-based approach to cell type labeling.  

CellTypist is described [on their website](https://www.celltypist.org/), and in this publication:  

Domínguez Conde, C. et al. Cross-tissue immune cell analysis reveals tissue-specific features in humans. Science 376, eabl5197 (2022)

Here, we'll load in our cells individually, and assign labels based on our 3-level annotated PBMC reference:  

- AIFI_L1:
    - 9 types
- AIFI_L2:  
    - 29 types
- AIFI_L3:
    - 71 types

## Load Packages

`anndata`: Data structures for scRNA-seq  
`celltypist`: Model-based cell type annotation  
`concurrent.futures`: parallelization methods  
`datetime`: date and time functions  
`h5py`: HDF5 file I/O  
`hisepy`: The HISE SDK for Python  
`numpy`: Mathematical data structures and computation  
`os`: operating system calls  
`pandas`: DataFrame data structures  
`re`: Regular expressions  
`scanpy`: scRNA-seq analysis  
`scipy.sparse`: Spare matrix data structures  
`shutil`: Shell utilities

In [1]:
import anndata
import celltypist
from celltypist import models
import concurrent.futures
from datetime import date
import h5py
import hisepy
import numpy as np
import os
import pandas as pd 
import re
import scanpy as sc
import scipy.sparse as scs
import shutil

Load a model to prevent CellTypist from loading all models per core

In [2]:
models.download_models(
    force_update = True,
    model = ['Immune_All_High.pkl']
)

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 48
📂 Storing models in /root/.celltypist/data/models
💾 Total models to download: 1
💾 Downloading model [1/1]: Immune_All_High.pkl


## Helper functions

This function allows easy reading of .csv files stored in HISE

In [3]:
def read_csv_uuid(csv_uuid):
    csv_path = '/home/jupyter/cache/{u}'.format(u = csv_uuid)
    if not os.path.isdir(csv_path):
        hise_res = hisepy.reader.cache_files([csv_uuid])
    csv_filename = os.listdir(csv_path)[0]
    csv_file = '{p}/{f}'.format(p = csv_path, f = csv_filename)
    df = pd.read_csv(csv_file, index_col = 0)
    return df

This function allows easy identification of the cached file path for files retrieved from HISE

In [4]:
def read_path_uuid(file_uuid):
    file_path = '/home/jupyter/cache/{u}'.format(u = file_uuid)
    if not os.path.isdir(file_path):
        hise_res = hisepy.reader.cache_files([file_uuid])
    filename = os.listdir(file_path)[0]
    full_path = '{p}/{f}'.format(p = file_path, f = filename)
    return full_path

These functions will retrieve data for a sample, assemble an AnnData object

In [5]:
# define a function to read count data
def read_mat(h5_con):
    mat = scs.csc_matrix(
        (h5_con['matrix']['data'][:], # Count values
         h5_con['matrix']['indices'][:], # Row indices
         h5_con['matrix']['indptr'][:]), # Pointers for column positions
        shape = tuple(h5_con['matrix']['shape'][:]) # Matrix dimensions
    )
    return mat

# define a function to read obeservation metadata (i.e. cell metadata)
def read_obs(h5con):
    bc = h5con['matrix']['barcodes'][:]
    bc = [x.decode('UTF-8') for x in bc]

    # Initialized the DataFrame with cell barcodes
    obs_df = pd.DataFrame({ 'barcodes' : bc })

    # Get the list of available metadata columns
    obs_columns = h5con['matrix']['observations'].keys()

    # For each column
    for col in obs_columns:
        # Read the values
        values = h5con['matrix']['observations'][col][:]
        # Check for byte storage
        if(isinstance(values[0], (bytes, bytearray))):
            # Decode byte strings
            values = [x.decode('UTF-8') for x in values]
        # Add column to the DataFrame
        obs_df[col] = values

    obs_df = obs_df.set_index('barcodes', drop = False)
    
    return obs_df

# define a function to construct anndata object from a h5 file
def read_h5_anndata(h5_con):
    #h5_con = h5py.File(h5_file, mode = 'r')
    # extract the expression matrix
    mat = read_mat(h5_con)
    # extract gene names
    genes = h5_con['matrix']['features']['name'][:]
    genes = [x.decode('UTF-8') for x in genes]
    # extract metadata
    obs_df = read_obs(h5_con)
    # construct anndata
    adata = anndata.AnnData(mat.T,
                             obs = obs_df)
    # make sure the gene names aligned
    adata.var_names = genes

    adata.var_names_make_unique()
    return adata

This function retrieves and assembles an anndata based on the UUID for a .h5 file.

In [6]:
def get_adata(uuid):
    # Load the file using HISE
    res = hisepy.reader.read_files([uuid])

    # If there's an error, read_files returns a list instead of a dictionary.
    # We should raise and exception with the message when this happens.
    if(isinstance(res, list)):
        error_message = res[0]['message']
        raise Exception('{u}: {e}'.format(u = uuid, e = error_message))
    
    # Read the file to adata
    h5_con = res['values'][0]
    adata = read_h5_anndata(h5_con)
    
    # Close the file now that we're done with it
    h5_con.close()

    return(adata)

This function applies cell type predictions for a specific model and generates output files.

In [7]:
def run_prediction(adata, model, model_name, out_dir = "output"):
    # Make output directories
    model_dir = "{d}/{m}".format(d = out_dir, m = model_name)
    if not os.path.isdir(model_dir):
        os.makedirs(model_dir)
    
    sample_id = adata.obs['pbmc_sample_id'].unique()[0]
    label_file = "{d}/{s}_{m}_labels.csv".format(d = model_dir, s = sample_id, m = model_name)

    if os.path.exists(label_file):
        print("{s}: {m} Previously computed; Skipping.".format(s = sample_id, m = model_name))
    else:
        # Perform prediction
        predictions = celltypist.annotate(
            adata, 
            model = model, 
            majority_voting = True)
    
        # Write output
        
        prob_file = "{d}/{s}_{m}_probability_mat.parquet".format(d = model_dir, s = sample_id, m = model_name)
        prob = predictions.probability_matrix
        prob.to_parquet(prob_file)
    
        dec_file = "{d}/{s}_{m}_decision_mat.parquet".format(d = model_dir, s = sample_id, m = model_name)
        predictions.decision_matrix.to_parquet(dec_file)
        
        labels = predictions.predicted_labels
        labels = labels.rename({'predicted_labels': model_name}, axis = 1)
        
        prob_scores = []
        for i in range(labels.shape[0]):
            prob_scores.append(prob.loc[labels.index.to_list()[i],labels[model_name][i]])
        labels['{m}_score'.format(m = model_name)] = prob_scores
        labels.to_csv(label_file)

This wrapper puts the above steps together: reading data based on a UUID, normalizing the data, and labeling with all of the models in model_paths

In [8]:
def process_data(file_uuid, sample_id, model_paths):
    out_dir = "output"
    check_file = '{d}/{m}/{s}_{m}_labels.csv'.format(d = out_dir, m = 'AIFI_L3', s = sample_id)

    if os.path.exists(check_file):
        print('{s} Previously labeled; Skipping.'.format(s = sample_id))
    else:
        # Load cells from HISE .h5 files
        adata = get_adata(file_uuid)
        
        # Normalize data
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        adata.obs.index = adata.obs['barcodes']
        
        # Predict cell types
        for model_name,model_path in model_paths.items():
            run_prediction(
                adata,
                model_path,
                model_name,
                out_dir
            )
        
        del adata

This function is used to generate a unique identifier for the notebook results to help with searching for them in HISE.

In [9]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Obtain CellTypist Models

In [10]:
model_uuids = {
    'AIFI_L1': '482b9ec5-8631-48a1-a7ef-94e23fe97068',
    'AIFI_L2': 'cc78f20a-b962-4ddf-ae93-47d58741a644',
    'AIFI_L3': '671d1e43-bd32-4fea-bdda-d19a0484e664'
}

In [11]:
model_paths = {}
for name,uuid in model_uuids.items():
    model_paths[name] = read_path_uuid(uuid)

downloading fileID: 482b9ec5-8631-48a1-a7ef-94e23fe97068
Files have been successfully downloaded!
downloading fileID: cc78f20a-b962-4ddf-ae93-47d58741a644
Files have been successfully downloaded!
downloading fileID: 671d1e43-bd32-4fea-bdda-d19a0484e664
Files have been successfully downloaded!


In [13]:
model_paths

{'AIFI_L1': '/home/jupyter/cache/482b9ec5-8631-48a1-a7ef-94e23fe97068/ref_pbmc_clean_celltypist_model_AIFI_L1_2024-04-18.pkl',
 'AIFI_L2': '/home/jupyter/cache/cc78f20a-b962-4ddf-ae93-47d58741a644/ref_pbmc_clean_celltypist_model_AIFI_L2_2024-04-19.pkl',
 'AIFI_L3': '/home/jupyter/cache/671d1e43-bd32-4fea-bdda-d19a0484e664/ref_pbmc_clean_celltypist_model_AIFI_L3_2024-04-19.pkl'}

## Read sample metadata from HISE

In [17]:
sample_meta_file_uuid = '2b3673ee-827b-415e-837b-f27a0b88eae2'
file_query = hisepy.reader.read_files(
    [sample_meta_file_uuid]
)

In [18]:
meta_data = file_query['values']

In [19]:
meta_data.shape

(512, 34)

## Apply across files

Here, we'll use `concurrent.futures` to apply the function above to our files in parallel.

In [20]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [21]:
file_uuids = meta_data['file.id'].to_list()
sample_ids = meta_data['sample.sampleKitGuid'].to_list()

In [22]:
model_paths

{'AIFI_L1': '/home/jupyter/cache/482b9ec5-8631-48a1-a7ef-94e23fe97068/ref_pbmc_clean_celltypist_model_AIFI_L1_2024-04-18.pkl',
 'AIFI_L2': '/home/jupyter/cache/cc78f20a-b962-4ddf-ae93-47d58741a644/ref_pbmc_clean_celltypist_model_AIFI_L2_2024-04-19.pkl',
 'AIFI_L3': '/home/jupyter/cache/671d1e43-bd32-4fea-bdda-d19a0484e664/ref_pbmc_clean_celltypist_model_AIFI_L3_2024-04-19.pkl'}

In [23]:
print(len(file_uuids))
print(len(sample_ids))

512
512


In [43]:
# Process each subset in parallel
pool_executor = concurrent.futures.ProcessPoolExecutor(max_workers = 62)
with pool_executor as executor:
    
    futures = []
    for i in range(len(file_uuids)):
        file_uuid = file_uuids[i]
        sample_id = sample_ids[i]
        futures.append(executor.submit(process_data, file_uuid, sample_id, model_paths)) ##added model_paths arg

    # Check for errors when parallel processes return results
    for future in concurrent.futures.as_completed(futures):
        try:
            print(future.result())
        except Exception as e:
            print(f'Error: {e}')

PB00069-06: AIFI_L1 Previously computed; Skipping.
PB00069-06: AIFI_L2 Previously computed; Skipping.
PB00069-06: AIFI_L3 Previously computed; Skipping.
None
PB02977-001: AIFI_L1 Previously computed; Skipping.
PB02977-001: AIFI_L2 Previously computed; Skipping.
PB02977-001: AIFI_L3 Previously computed; Skipping.
None
PB03933-001: AIFI_L1 Previously computed; Skipping.
PB03933-001: AIFI_L2 Previously computed; Skipping.
PB03933-001: AIFI_L3 Previously computed; Skipping.
None
PB04463-001: AIFI_L1 Previously computed; Skipping.
PB04463-001: AIFI_L2 Previously computed; Skipping.
PB04463-001: AIFI_L3 Previously computed; Skipping.
PB02955-001: AIFI_L1 Previously computed; Skipping.
PB02955-001: AIFI_L2 Previously computed; Skipping.
PB02955-001: AIFI_L3 Previously computed; Skipping.
None
None
PB00475-01: AIFI_L1 Previously computed; Skipping.
PB00475-01: AIFI_L2 Previously computed; Skipping.
PB00475-01: AIFI_L3 Previously computed; Skipping.
None
PB02983-001: AIFI_L1 Previously computed

🔬 Input data has 14336 cells and 33538 genes
🔗 Matching reference genes in the model


PB04638-001: AIFI_L1 Previously computed; Skipping.
PB04638-001: AIFI_L2 Previously computed; Skipping.
PB04638-001: AIFI_L3 Previously computed; Skipping.
PB04470-002: AIFI_L1 Previously computed; Skipping.
PB04470-002: AIFI_L2 Previously computed; Skipping.
PB04470-002: AIFI_L3 Previously computed; Skipping.
PB04654-001: AIFI_L1 Previously computed; Skipping.
PB04654-001: AIFI_L2 Previously computed; Skipping.
PB04654-001: AIFI_L3 Previously computed; Skipping.
None
None
None
PB02134-09: AIFI_L1 Previously computed; Skipping.
PB02134-09: AIFI_L2 Previously computed; Skipping.
PB02134-09: AIFI_L3 Previously computed; Skipping.
PB04660-002: AIFI_L1 Previously computed; Skipping.
PB04660-002: AIFI_L2 Previously computed; Skipping.
PB04660-002: AIFI_L3 Previously computed; Skipping.
None
None
None


🧬 1109 features used for prediction
⚖️ Scaling input data


PB04094-002: AIFI_L1 Previously computed; Skipping.
PB04094-002: AIFI_L2 Previously computed; Skipping.
PB04094-002: AIFI_L3 Previously computed; Skipping.


...

PB00788-01: AIFI_L1 Previously computed; Skipping.
PB00788-01: AIFI_L2 Previously computed; Skipping.
PB00788-01: AIFI_L3 Previously computed; Skipping.


🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 10


None
PB00078-01: AIFI_L1 Previously computed; Skipping.
PB00078-01: AIFI_L2 Previously computed; Skipping.
PB00078-01: AIFI_L3 Previously computed; Skipping.
PB00451-01: AIFI_L1 Previously computed; Skipping.
PB00451-01: AIFI_L2 Previously computed; Skipping.
PB00451-01: AIFI_L3 Previously computed; Skipping.
PB00080-01: AIFI_L1 Previously computed; Skipping.
PB00080-01: AIFI_L2 Previously computed; Skipping.
PB00080-01: AIFI_L3 Previously computed; Skipping.
PB00099-01: AIFI_L1 Previously computed; Skipping.
PB00099-01: AIFI_L2 Previously computed; Skipping.
PB00099-01: AIFI_L3 Previously computed; Skipping.
None
None
None
None
PB02847-001: AIFI_L1 Previously computed; Skipping.
PB02847-001: AIFI_L2 Previously computed; Skipping.
PB02847-001: AIFI_L3 Previously computed; Skipping.
PB00094-01: AIFI_L1 Previously computed; Skipping.
PB00094-01: AIFI_L2 Previously computed; Skipping.
PB00094-01: AIFI_L3 Previously computed; Skipping.
None
None
PB00784-01: AIFI_L1 Previously computed; Ski

🗳️ Majority voting the predictions
✅ Majority voting done!


PB00225-01: AIFI_L1 Previously computed; Skipping.
PB00225-01: AIFI_L2 Previously computed; Skipping.
PB00225-01: AIFI_L3 Previously computed; Skipping.
PB00228-01: AIFI_L1 Previously computed; Skipping.
PB00228-01: AIFI_L2 Previously computed; Skipping.
PB00228-01: AIFI_L3 Previously computed; Skipping.
PB00226-01: AIFI_L1 Previously computed; Skipping.
PB00226-01: AIFI_L2 Previously computed; Skipping.
PB00226-01: AIFI_L3 Previously computed; Skipping.
PB00219-01: AIFI_L1 Previously computed; Skipping.
None
None
None
PB00219-01: AIFI_L2 Previously computed; Skipping.
PB00219-01: AIFI_L3 Previously computed; Skipping.
PB00097-01: AIFI_L1 Previously computed; Skipping.
PB00097-01: AIFI_L2 Previously computed; Skipping.
PB00097-01: AIFI_L3 Previously computed; Skipping.
PB00093-01: AIFI_L1 Previously computed; Skipping.
PB00093-01: AIFI_L2 Previously computed; Skipping.
PB00093-01: AIFI_L3 Previously computed; Skipping.
None
None
None
PB00206-01: AIFI_L1 Previously computed; Skipping.
P

## Assemble results

For each model, we'll assemble the results as a .csv file that we can utilize later for subclustering and analysis of major cell classes.

In [44]:
models = list(model_paths.keys())

In [45]:
out_files = []
for model in models:
    model_path = 'output/{m}'.format(m = model)
    model_path_files = os.listdir(model_path)
    model_files = []
    for model_path_file in model_path_files:
        if 'labels' in model_path_file:
            model_files.append(model_path_file)
    
    model_list = []
    for model_file in model_files:
        df = pd.read_csv('output/{m}/{f}'.format(m = model, f = model_file))
        model_list.append(df)
    model_df = pd.concat(model_list)

    out_csv = 'output/ra_celltypist_{m}_{d}.csv'.format(
        m = model, d = date.today())
    out_files.append(out_csv)
    
    model_df.to_csv(out_csv)

    out_parquet = 'output/ra_celltypist_{m}_{d}.parquet'.format(
        m = model, d = date.today())
    out_files.append(out_parquet)
    
    model_df.to_parquet(out_parquet)

In [46]:
model_df.shape

(8519746, 5)

In [47]:
model_df.head()

,barcodes,AIFI_L3,over_clustering,majority_voting,AIFI_L3_score
0,b7fa1ede1cd011eeb16b46ccef6ada21,Core naive B cell,0,Core naive B cell,1.0
1,b7fa214a1cd011eeb16b46ccef6ada21,Core naive CD4 T cell,32,Core naive CD4 T cell,1.0
2,b7fa2b861cd011eeb16b46ccef6ada21,Core CD14 monocyte,26,Core CD14 monocyte,1.0
3,b7fa2bc21cd011eeb16b46ccef6ada21,GZMK- CD56dim NK cell,71,GZMK- CD56dim NK cell,1.0
4,b7fa30fe1cd011eeb16b46ccef6ada21,Core naive CD4 T cell,127,Core naive CD4 T cell,1.0


## Upload assembled data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [48]:
ss = hisepy.get_study_spaces()

In [49]:
print(ss[2]['name'])
print(ss[2]['id'])
study_space_uuid = ss[2]['id']

title = '01 Raw scRNA-seq Assembly {d}'.format(d = date.today())

preRA cross-sectional + earlyRA + Longitudinal analysis
223de760-9624-45bd-aefe-ca24c75b1800


In [35]:
title = '01 CellTypist Label Results {d}'.format(d = date.today())

In [50]:
search_id = element_id()
search_id

'zinc-bismuth-bismuth'

In [51]:
in_files = list(model_uuids.values()) + [sample_meta_file_uuid] + meta_data['file.id'].to_list() 

In [52]:
in_files[0:10]

['482b9ec5-8631-48a1-a7ef-94e23fe97068',
 'cc78f20a-b962-4ddf-ae93-47d58741a644',
 '671d1e43-bd32-4fea-bdda-d19a0484e664',
 '2b3673ee-827b-415e-837b-f27a0b88eae2',
 '41caa9d3-12e4-432b-8352-aa012905b823',
 'd5720dff-ea39-4b45-9e45-b7459b779ee6',
 'a2829968-97f2-48c8-bc0a-2989850aebf1',
 'd75a9be9-7092-4915-8532-b75c99a885fe',
 '6b48b8b3-6848-4f2f-bc63-c2ccd7819948',
 '7e8f9b5f-0af6-4b8e-94e3-51027891dca0']

In [53]:
len(in_files)

516

In [54]:
out_files

['output/ra_celltypist_AIFI_L1_2024-05-10.csv',
 'output/ra_celltypist_AIFI_L1_2024-05-10.parquet',
 'output/ra_celltypist_AIFI_L2_2024-05-10.csv',
 'output/ra_celltypist_AIFI_L2_2024-05-10.parquet',
 'output/ra_celltypist_AIFI_L3_2024-05-10.csv',
 'output/ra_celltypist_AIFI_L3_2024-05-10.parquet']

In [55]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/ra_celltypist_AIFI_L1_2024-05-10.csv', 'output/ra_celltypist_AIFI_L1_2024-05-10.parquet', 'output/ra_celltypist_AIFI_L2_2024-05-10.csv', 'output/ra_celltypist_AIFI_L2_2024-05-10.parquet', 'output/ra_celltypist_AIFI_L3_2024-05-10.csv', 'output/ra_celltypist_AIFI_L3_2024-05-10.parquet']. Do you truly want to proceed?


(y/n) y


{'trace_id': 'eb4a2f0d-910c-4885-9b5a-f5f249e1763c',
 'files': ['output/ra_celltypist_AIFI_L1_2024-05-10.csv',
  'output/ra_celltypist_AIFI_L1_2024-05-10.parquet',
  'output/ra_celltypist_AIFI_L2_2024-05-10.csv',
  'output/ra_celltypist_AIFI_L2_2024-05-10.parquet',
  'output/ra_celltypist_AIFI_L3_2024-05-10.csv',
  'output/ra_celltypist_AIFI_L3_2024-05-10.parquet']}

In [56]:
import session_info
session_info.show()

#### Notes
1. hisepy cache_files vs read_files?
2. check number of labeled samples at the end of parallel processing
3. majority voting = True?
4. rename majority voting columns based on label level